<div align="right"><i>Matías Torres Esteban<br>Diciembre, 2025</i></div>

# Interpretación de Textos con LLMs

Los grandes modelos de lenguaje (LLMs) son conocidos por su capacidad de aprender patrones complejos de los textos y de resolver en consecuencia varias tareas de procesamiento del lenguaje natural, como lo son la traducción automática, la generación de resúmenes y la resolución de preguntas de dominios especializados. Es por esto que han surgido nuevas lineas de investigación que buscan utilizar a los grandes modelos de lenguaje (LLMs) como *interpretadores semánticos de textos*. Su objetivo es representar el significado de un texto escrito en lenguaje natural en uno o más lenguajes lógicos —por ejemplo, la lógica proposicional o la lógica de primer orden— para explicitar la información contenida allí y brindarle mayor estructura. De esta manera, podemos obtener representaciones avanzadas del significado de un texto que sean más faciles de almacenar en una computadora y puedan ser manipuladas algorítmicamente.

Un lenguaje lógico que podemos utilizar para representar la información contenida en un texto es el lenguaje de los *Grafos de Conocimiento (KGs)*, los cuales permiten codificar hechos y proposiciones del mundo mediante ternas $(f,r,o)$ donde $f$ denota un concepto fuente, $r$ una relación semántica y $o$ un concepto objetivo. Así, la información contenida en el siguiente texto:

* *"El Covid-19 es una enfermedad infecciosa causada por el SARS-CoV-2. Produce síntomas que incluyen fiebre, tos y fatiga"*,

puede representarse como la siguiente colección de ternas de conocimiento:

* *(Covid-19, es una, Enfermedad Infecciosa)*
* *(Covid-19, causada por, SARS-CoV-2)*.
* *(Covid-19, tiene síntoma, Fiebre)*.
* *(Covid-19, tiene síntoma, Tos)*.
* *(Covid-19, tiene síntoma, Fatiga)*.

Esta colección de ternas puede representase como un grafo dirigido y etiquetado, como se muestra en la siguiente figura:

![GrafoConocimiento](https://raw.githubusercontent.com/matizzat/InforSanLuis-2025-LLMs/5596c0c724cd8e63742866ad998479ecd6f685d2/imagenes/grafo_conocimiento_covid19.svg)

Vemos que este tipo de representación es más rica que el texto puro porque explicita los conceptos y relaciones más relevantes. De esta manera, la interpretación y el análisis de la información extraída se vuelven más accesibles tanto para un usuario como para una computadora, y además se facilita su almacenamiento en bases de datos. Finalmente, si pudiéramos procesar automáticamente un gran conjunto de documentos de un dominio especializado —como la biología o la medicina— y transformarlos en grafos de conocimiento, podríamos aplicar todo el aparato matemático de la Teoría de Grafos para analizar estos sistemas conceptuales y revelar información del dominio que está escondida en los textos.

En esta notebook utilizaremo al modelo Gemini junto a código Python para crear automáticamente grafos de conocimiento a partir de textos. Este ejercicio nos enseñará a coordinar diferentes invocaciones al modelo y a escribir correctamente nuestros prompts para sacarle el máximo provecho posible.

## Proceso de Interpretación

Vamos a implementar un procedimiento de interpretación textual inspirado en la metodología propuesta por Joseph Novak para la construcción de mapas conceptuales [1]. Este procedimiento permite generar un KG a partir de un texto expositivo mediante una secuencia estructurada de 4 pasos:

1. Solicitamos al modelo que analice el texto ``<texto>`` y genere una pregunta de enfoque ``<pregunta>``. Las ternas de conocimiento generadas en los
próximos pasos deberı́an ayudar a responder esta pregunta.

2. Solicitamos al modelo que analice ``<texto>`` y ``<pregunta>`` y que luego genere una lista de conceptos ``<conceptos>``. Los conceptos extraı́dos deben
estar explicitamente mencionados en el texto y tienen que ayudar a responder la pregunta de enfoque.

3. Solictamos al modelo que analice ``<texto>``, ``<pregunta>`` y ``<conceptos>`` y que genere una lista de relaciones semánticas ``<relaciones>``.

4. Solicitamos al modelo que analice ``<texto>``, ``<pregunta>``, ``<conceptos>`` y ``<relaciones>`` en conjunto y que luego genere una lista de ternas de conocimiento ``<ternas>``.

En cada paso realizamos un procesamiento de las etiquetas de conceptos y relaciones para convertir todos sus caracteres a minúsculas y eliminar espacios en blanco innecesarios.

## Código

**Advertencia:** Si la API de Gemini no está disponible en este momento pueden simular el proceso en ChatGPT o el chat de Gemini y reintentar ejecutar el código más tarde.

Primero extraemos los recursos desde Github (textos y prompts):

In [1]:
!git clone https://github.com/matizzat/InforSanLuis-2025-LLMs
%cd InforSanLuis-2025-LLMs

Cloning into 'InforSanLuis-2025-LLMs'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 118 (delta 57), reused 31 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 758.78 KiB | 3.26 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/InforSanLuis-2025-LLMs


Instalamos la librería Pyvis para visualizar grafos de conocimiento:

In [7]:
!pip install pyvis

Importamos las librerías necesarias

In [8]:
from pyvis.network import Network
from google.colab import userdata
from pyvis import network as net
from google.genai import types
from google import genai
from typing import List
import networkx as nx
import pprint
import json
import re

Instanciamos un cliente para invocar al modelo Gemini. Para ejecutar esta celda deben obtener una clave del siguiente [enlace](https://ai.google.dev/gemini-api/docs/api-key). También tienen que configurar Google Colab para utilizar esta clave (Ver  [Tutorial](https://www.youtube.com/watch?v=snrvP_TZjvw))

In [9]:
GOOGLE_API_KEY = userdata.get('GEMINI_KEY_API')
cliente_genai = genai.Client(api_key=GOOGLE_API_KEY)

Definimos una función auxiliar para abrir el contenido de un archivo y cargamos las instrucciones con las que invocaremos al modelo. En cada invocación nosotros le proveemos al modelo una **instrucción de sistema** y una **instrucción de usuario**:

* En la instrucción de sistema nosotros le especificamos al modelo cual es el rol que debe cumplir, le explicamos en términos generales la tarea a resolver y le brindamos ejemplos concretos de cómo hay que resolverla y cual es el formato de salida esperado.

* En la instrucción de usuario especificamos cual es el texto corriente que queremos analizar.

La técnica donde le damos ejemplos concretos al modelo de cómo debe resolver una tarea se denomina **aprendizaje en contexto** y es muy importante aprenderla para embeber a los LLMs en procesos automáticos y aplicarlos en dominios especializados.

In [10]:
def abrir_contenido_archivo(nombre_archivo: str):
    with open(nombre_archivo, 'r') as f:
        return f.read()

crear_pregunta_sistema   = abrir_contenido_archivo('./instrucciones/crear_pregunta_sistema.txt')
crear_conceptos_sistema  = abrir_contenido_archivo('./instrucciones/crear_conceptos_sistema.txt')
crear_relaciones_sistema = abrir_contenido_archivo('./instrucciones/crear_relaciones_sistema.txt')
crear_ternas_sistema     = abrir_contenido_archivo('./instrucciones/crear_ternas_sistema.txt')

crear_pregunta_usuario   = abrir_contenido_archivo('./instrucciones/crear_pregunta_usuario.txt')
crear_conceptos_usuario  = abrir_contenido_archivo('./instrucciones/crear_conceptos_usuario.txt')
crear_relaciones_usuario = abrir_contenido_archivo('./instrucciones/crear_relaciones_usuario.txt')
crear_ternas_usuario     = abrir_contenido_archivo('./instrucciones/crear_ternas_usuario.txt')

Ejecuten la siguiente celda probando distintos valores para visualizar las instrucciones con las que vamos a invocar al modelo. Observar que las instrucciones del usuario funcionan como plantillas: las completamos dinámicamente a medida que el LLM avance en cada paso del proceso. Los espacios a modificar dinámicamente están encerrados entre llaves ``{`` y ``}``.

In [ ]:
print(crear_relaciones_sistema)

In [ ]:
print(crear_pregunta_sistema)

In [ ]:
print(crear_conceptos_sistema)

In [ ]:
print(crear_conceptos_usuario)

In [ ]:
print(crear_pregunta_usuario)

A continuación definimos funciones auxiliares que necesitaremos en nuestro proceso:
* ``invocar_llm`` recibe como parámetros una instrucción de sistema y una instrucción de usuario y con ellos invoca al modelo Gemini 2.5 Flash. La invocación esta fijada con un valor de temperatura bajo para que las respuestas del modelo sean determinísticas y no varíen en diferentes invocaciones.  
* ``abrir_lote_de_documentos`` se encarga de abrir una lista de documentos desde la ruta especificada. Cada documento posee un título y un texto.
* ``guardar_mapa_conceptuales``: Almacena una lista de mapas conceptuales en un archivo JSON. Cada mapa conceptual está codificado como un diccionario que tiene los siguientes elementos:
  * ``titulo``: Un string con el título del mapa conceptual para identificarlo.
  * ``pregunta``: Un string que representa la pregunta focal.
  * ``conceptos``: Una lista de strings que representa las etiquetas de conceptos.
  * ``relaciones``: Una lista de strings que representa las etiquetas de las relaciones semánticas.
  * ``ternas``. Una lista de ternas de conocimiento, donde cada una es un diccionario con las componentes `f` (concepto fuente), `r` (relación) y `o` (concepto objetivo).
* ``dibujar_mapas_conceptuales``: Dibuja un conjunto de mapas conceptuales como un único grafo dirigido y etiquetado, el cual se almacena en un archivo HTML.

In [11]:
def invocar_llm(sistema: str, usuario: str):
    """
    Invoca al modelo de lenguaje Gemini 2.5 Flash.

    Parámetros:
        sistema (str): Instrucciones del sistema para el modelo.
        usuario  (str): Consulta o instrucciones del usuario.

    Retorna:
        str: Texto generado por el modelo.
    """
    global cliente_genai

    respuesta = cliente_genai.models.generate_content(
        config = types.GenerateContentConfig(
            system_instruction = sistema,
            temperature = 0.1
        ),
        model = "gemini-2.5-flash",
        contents = usuario
    )

    return respuesta.text


def abrir_lote_de_documentos(ruta_entrada: str) -> list[dict]:
    """
    Abre un lote de documentos desde un archivo JSON.

    Parámetros:
        ruta_entrada (str): Ruta del archivo que contiene los documentos.

    Retorna:
        list[dict]: Lista de documentos.
    """
    with open(ruta_entrada, 'r') as f:
        textos = json.load(f)
    return textos


def guardar_mapas_conceptuales(ruta_salida: str, mapas: dict):
    """
    Guarda una lista de mapas conceptuales en un archivo JSON.

    Parámetros:
        ruta_salida (str): Ruta del archivo de salida.
        mapas (dict): Mapas conceptuales a guardar.
    """
    with open(ruta_salida, 'w') as f:
        json.dump(mapas, f, indent=4)


def dibujar_mapas_conceptuales(ruta_salida: str, mapas: dict):
    """
    Crea una visualización pyvis de una lista de mapas conceptuales
    y la guarda como archivo HTML.

    Parámetros:
        mapas (dict): Lista de mapas conceptuales.
        ruta_salida (str): Archivo HTML donde se guardará la visualización.
    """
    G = nx.MultiDiGraph()

    for mapa in mapas:
        pregunta_focal = mapa['pregunta']
        ternas = mapa['ternas']

        for terna in ternas:
            f = terna['f']
            r = terna['r']
            o = terna['o']

            if f not in G:
                G.add_node(f, label=f, color='black')
            if o not in G:
                G.add_node(o, label=o, color='black')

            G.add_edge(
                f,
                o,
                label=r,
                title=pregunta_focal,
                color='blue'
            )

    vis = net.Network(directed=True)
    vis.from_nx(G)
    vis.save_graph(ruta_salida)


Abrimos el lote de documentos y mostramos el texto del primer documento que está almacenado allí. Si revisan el archivo `documentos.json` verán que cada documento consiste de un título y un texto.

In [33]:
documentos = abrir_lote_de_documentos('./datos/documentos.json')
texto = documentos[4]['texto']
print(texto)

El ojo es un órgano que detecta la luz y es la base del sentido de la vista. Su función consiste básicamente en transformar la energía lumínica en señales eléctricas que son enviadas al cerebro a través del nervio óptico.
Funciona de forma muy similar al de la mayoría de los vertebrados y algunos moluscos; posee una lente llamada cristalino, que es ajustable según la distancia; un diafragma, que se llama pupila, cuyo diámetro está regulado por el iris, y un tejido sensible a la luz, que es la retina.
La luz penetra a través de la pupila, atraviesa el cristalino y se proyecta sobre la retina, donde se transforma, gracias a unas células llamadas fotorreceptoras, en impulsos nerviosos que se trasladan, a través del nervio óptico, al cerebro.



Definimos una función auxiliar para normalizar la etiqueta de un concepto o una relación semántica. Esta elimina el carácter especial `@`, quita los espacios en blanco innecesarios y convierte todas las letras a minúsculas

In [34]:
def normalizar_etiqueta(etiqueta: str):
    etiqueta = etiqueta.replace('@','')
    etiqueta = etiqueta.strip()
    etiqueta = " ".join(etiqueta.split())
    etiqueta = etiqueta.lower()
    return etiqueta

Formateamos el prompt de usuario para instruirle al modelo que genere una pregunta de enfoque.

In [35]:
# Paso 1
instruccion_crear_pregunta_formateada = crear_pregunta_usuario.format(texto = texto)
print(instruccion_crear_pregunta_formateada )

Texto de Conocimiento:
@
El ojo es un órgano que detecta la luz y es la base del sentido de la vista. Su función consiste básicamente en transformar la energía lumínica en señales eléctricas que son enviadas al cerebro a través del nervio óptico.
Funciona de forma muy similar al de la mayoría de los vertebrados y algunos moluscos; posee una lente llamada cristalino, que es ajustable según la distancia; un diafragma, que se llama pupila, cuyo diámetro está regulado por el iris, y un tejido sensible a la luz, que es la retina.
La luz penetra a través de la pupila, atraviesa el cristalino y se proyecta sobre la retina, donde se transforma, gracias a unas células llamadas fotorreceptoras, en impulsos nerviosos que se trasladan, a través del nervio óptico, al cerebro.

@
Pregunta Focal:



Invocamos al modelo de lenguaje y le solicitamos que cree una pregunta focal para el texto dado.

In [36]:
pregunta_focal = invocar_llm(
    usuario = instruccion_crear_pregunta_formateada,
    sistema = crear_pregunta_sistema)
print(pregunta_focal)
# Coloca todo en miniscula y elimina los espacios en blanco
pregunta_focal = normalizar_etiqueta(pregunta_focal)
print(pregunta_focal)

¿Qué es el ojo, cuál es su función y cómo funciona?
¿qué es el ojo, cuál es su función y cómo funciona?


Formateamos el prompt de usuario para instruirle al modelo que genere una lista de conceptos.

In [37]:
# Paso 2
instruccion_crear_conceptos_formateada = crear_conceptos_usuario.format(pregunta = pregunta_focal, texto = texto)
print(instruccion_crear_conceptos_formateada)

Texto de Conocimiento:
@
Pregunta Focal:
¿qué es el ojo, cuál es su función y cómo funciona?
Conocimiento:
El ojo es un órgano que detecta la luz y es la base del sentido de la vista. Su función consiste básicamente en transformar la energía lumínica en señales eléctricas que son enviadas al cerebro a través del nervio óptico.
Funciona de forma muy similar al de la mayoría de los vertebrados y algunos moluscos; posee una lente llamada cristalino, que es ajustable según la distancia; un diafragma, que se llama pupila, cuyo diámetro está regulado por el iris, y un tejido sensible a la luz, que es la retina.
La luz penetra a través de la pupila, atraviesa el cristalino y se proyecta sobre la retina, donde se transforma, gracias a unas células llamadas fotorreceptoras, en impulsos nerviosos que se trasladan, a través del nervio óptico, al cerebro.

@
Lista de Conceptos:



Invocamos al modelo de lenguaje para que genere una lista de conceptos a partir del texto y la pregunta de enfoque:

In [38]:
respuesta_llm = invocar_llm(
    usuario = instruccion_crear_conceptos_formateada,
    sistema = crear_conceptos_sistema)

print(respuesta_llm)

Lista de Conceptos:
Ojo
Órgano que detecta la luz
Base del sentido de la vista
Sentido de la vista
Función del ojo
Transformar energía lumínica en señales eléctricas
Energía lumínica
Señales eléctricas
Cerebro
Nervio óptico
Funcionamiento del ojo
Vertebrados
Moluscos
Lente
Cristalino
Lente ajustable según la distancia
Diafragma
Pupila
Diámetro de la pupila
Iris
Tejido sensible a la luz
Retina
Luz
Células fotorreceptoras
Impulsos nerviosos


Convertimos el texto extraido en una lista de Python:

In [39]:
# Expresión regular para extraer una lista de conceptos:
conceptos_exp_reg = r'[\w\d].*?\n|[\w\d].*$'

lista_conceptos = re.findall(conceptos_exp_reg, respuesta_llm)
lista_conceptos = [normalizar_etiqueta(concepto) for concepto in lista_conceptos]

pprint.pprint(lista_conceptos)

['lista de conceptos:',
 'ojo',
 'órgano que detecta la luz',
 'base del sentido de la vista',
 'sentido de la vista',
 'función del ojo',
 'transformar energía lumínica en señales eléctricas',
 'energía lumínica',
 'señales eléctricas',
 'cerebro',
 'nervio óptico',
 'funcionamiento del ojo',
 'vertebrados',
 'moluscos',
 'lente',
 'cristalino',
 'lente ajustable según la distancia',
 'diafragma',
 'pupila',
 'diámetro de la pupila',
 'iris',
 'tejido sensible a la luz',
 'retina',
 'luz',
 'células fotorreceptoras',
 'impulsos nerviosos']


Formateamos el prompt de usuario para instruirle al modelo que genere una lista de relaciones semánticas:

In [40]:
# Paso 3
instruccion_crear_relaciones_formateada = crear_relaciones_usuario.format(
    pregunta = pregunta_focal,
    conceptos = "\n".join(lista_conceptos),
    texto = texto)
print(instruccion_crear_relaciones_formateada)

Texto de Conocimiento:
@
Pregunta Focal:
¿qué es el ojo, cuál es su función y cómo funciona?
Lista de Conceptos:
lista de conceptos:
ojo
órgano que detecta la luz
base del sentido de la vista
sentido de la vista
función del ojo
transformar energía lumínica en señales eléctricas
energía lumínica
señales eléctricas
cerebro
nervio óptico
funcionamiento del ojo
vertebrados
moluscos
lente
cristalino
lente ajustable según la distancia
diafragma
pupila
diámetro de la pupila
iris
tejido sensible a la luz
retina
luz
células fotorreceptoras
impulsos nerviosos
Conocimiento:
El ojo es un órgano que detecta la luz y es la base del sentido de la vista. Su función consiste básicamente en transformar la energía lumínica en señales eléctricas que son enviadas al cerebro a través del nervio óptico.
Funciona de forma muy similar al de la mayoría de los vertebrados y algunos moluscos; posee una lente llamada cristalino, que es ajustable según la distancia; un diafragma, que se llama pupila, cuyo diámetro 

Invocamos al modelo de lenguaje para que genere una lista de relaciones a partir del texto, la pregunta de enfoque y la lista de conceptos:

In [41]:
respuesta_llm = invocar_llm(
    sistema = crear_relaciones_sistema,
    usuario = instruccion_crear_relaciones_formateada)

print(respuesta_llm)

Relaciones Semánticas:
Es un
Es la base del
Consiste básicamente en transformar
En
Son enviadas al
A través del
Similar al de
Posee una
Llamada
Es
Se llama
Está regulado por el
Es la
Penetra a través de la
Atraviesa el
Se proyecta sobre la
Se transforma
Gracias a unas
Se trasladan
Al


Convertimos el texto extraido en una lista de Python:

In [42]:
# Expresión regular para extraer una lista de relaciones semánticas:
relaciones_exp_reg = r'[\w\d].*?\n|[\w\d].*$'

lista_relaciones = re.findall(relaciones_exp_reg, respuesta_llm)
lista_relaciones = [normalizar_etiqueta(relacion) for relacion in lista_relaciones]

pprint.pprint(lista_relaciones)
# Mejora: No deberia aparecer "relaciones semanticas" dentro de la Lista de relaciones semanticas

['relaciones semánticas:',
 'es un',
 'es la base del',
 'consiste básicamente en transformar',
 'en',
 'son enviadas al',
 'a través del',
 'similar al de',
 'posee una',
 'llamada',
 'es',
 'se llama',
 'está regulado por el',
 'es la',
 'penetra a través de la',
 'atraviesa el',
 'se proyecta sobre la',
 'se transforma',
 'gracias a unas',
 'se trasladan',
 'al']


Formateamos el prompt de usuario para instruirle al modelo que genere una lista de ternas de conocimiento:

In [43]:
instruccion_crear_ternas_formateada = crear_ternas_usuario.format(
    pregunta = pregunta_focal + "\n",
    conceptos = "\n".join(lista_conceptos) + "\n",
    relaciones = "\n".join(lista_relaciones),
    texto = texto)

print(instruccion_crear_ternas_formateada)

Texto de Conocimiento:
@
Pregunta Focal:
¿qué es el ojo, cuál es su función y cómo funciona?

Lista de Conceptos:
lista de conceptos:
ojo
órgano que detecta la luz
base del sentido de la vista
sentido de la vista
función del ojo
transformar energía lumínica en señales eléctricas
energía lumínica
señales eléctricas
cerebro
nervio óptico
funcionamiento del ojo
vertebrados
moluscos
lente
cristalino
lente ajustable según la distancia
diafragma
pupila
diámetro de la pupila
iris
tejido sensible a la luz
retina
luz
células fotorreceptoras
impulsos nerviosos

Lista de Relaciones:
relaciones semánticas:
es un
es la base del
consiste básicamente en transformar
en
son enviadas al
a través del
similar al de
posee una
llamada
es
se llama
está regulado por el
es la
penetra a través de la
atraviesa el
se proyecta sobre la
se transforma
gracias a unas
se trasladan
al
Conocimiento:
El ojo es un órgano que detecta la luz y es la base del sentido de la vista. Su función consiste básicamente en transforma

Invocamos al modelo de lenguaje para que genere una lista de ternas de conocimiento  a partir del texto, la pregunta de enfoque, la lista de conceptos y la lista de relaciones semánticas: # Paso 4

In [44]:
respuesta_llm = invocar_llm(
    sistema = crear_ternas_sistema,
    usuario = instruccion_crear_ternas_formateada)

print(respuesta_llm)

@! ojo @ es un @ órgano que detecta la luz !@
@! ojo @ es la base del @ sentido de la vista !@
@! función del ojo @ consiste básicamente en transformar @ energía lumínica !@
@! energía lumínica @ en @ señales eléctricas !@
@! señales eléctricas @ son enviadas al @ cerebro !@
@! señales eléctricas @ a través del @ nervio óptico !@
@! funcionamiento del ojo @ similar al de @ vertebrados !@
@! funcionamiento del ojo @ similar al de @ moluscos !@
@! ojo @ posee una @ lente !@
@! lente @ llamada @ cristalino !@
@! cristalino @ es @ lente ajustable según la distancia !@
@! ojo @ posee una @ diafragma !@
@! diafragma @ se llama @ pupila !@
@! diámetro de la pupila @ está regulado por el @ iris !@
@! ojo @ posee una @ tejido sensible a la luz !@
@! tejido sensible a la luz @ es la @ retina !@
@! luz @ penetra a través de la @ pupila !@
@! luz @ atraviesa el @ cristalino !@
@! luz @ se proyecta sobre la @ retina !@
@! luz @ se transforma @ impulsos nerviosos !@
@! luz @ en @ impulsos nerviosos 

Extraemos las ternas de conocimiento del modelo:

In [45]:
ternas_exp_reg = r'@! (.+?) @ (.+?) @ (.+?) !@'

L = re.findall(ternas_exp_reg, respuesta_llm)

lista_ternas = []

for f, r, o in L:
    lista_ternas.append({
        'f': normalizar_etiqueta(f),
        'r': normalizar_etiqueta(r),
        'o': normalizar_etiqueta(o)
    })

pprint.pprint(lista_ternas)

[{'f': 'ojo', 'o': 'órgano que detecta la luz', 'r': 'es un'},
 {'f': 'ojo', 'o': 'sentido de la vista', 'r': 'es la base del'},
 {'f': 'función del ojo',
  'o': 'energía lumínica',
  'r': 'consiste básicamente en transformar'},
 {'f': 'energía lumínica', 'o': 'señales eléctricas', 'r': 'en'},
 {'f': 'señales eléctricas', 'o': 'cerebro', 'r': 'son enviadas al'},
 {'f': 'señales eléctricas', 'o': 'nervio óptico', 'r': 'a través del'},
 {'f': 'funcionamiento del ojo', 'o': 'vertebrados', 'r': 'similar al de'},
 {'f': 'funcionamiento del ojo', 'o': 'moluscos', 'r': 'similar al de'},
 {'f': 'ojo', 'o': 'lente', 'r': 'posee una'},
 {'f': 'lente', 'o': 'cristalino', 'r': 'llamada'},
 {'f': 'cristalino', 'o': 'lente ajustable según la distancia', 'r': 'es'},
 {'f': 'ojo', 'o': 'diafragma', 'r': 'posee una'},
 {'f': 'diafragma', 'o': 'pupila', 'r': 'se llama'},
 {'f': 'diámetro de la pupila', 'o': 'iris', 'r': 'está regulado por el'},
 {'f': 'ojo', 'o': 'tejido sensible a la luz', 'r': 'posee 

Unimos todas las componentes obtenidas en un único diccionario Python que representa el grafo de conocimiento (o mapa conceptual):

In [46]:
mapa_conceptual = {'titulo': documentos[4]['titulo'], 'pregunta': pregunta_focal, 'ternas': lista_ternas, 'conceptos': lista_conceptos, 'relaciones': lista_relaciones}
pprint.pprint(mapa_conceptual)

{'conceptos': ['lista de conceptos:',
               'ojo',
               'órgano que detecta la luz',
               'base del sentido de la vista',
               'sentido de la vista',
               'función del ojo',
               'transformar energía lumínica en señales eléctricas',
               'energía lumínica',
               'señales eléctricas',
               'cerebro',
               'nervio óptico',
               'funcionamiento del ojo',
               'vertebrados',
               'moluscos',
               'lente',
               'cristalino',
               'lente ajustable según la distancia',
               'diafragma',
               'pupila',
               'diámetro de la pupila',
               'iris',
               'tejido sensible a la luz',
               'retina',
               'luz',
               'células fotorreceptoras',
               'impulsos nerviosos'],
 'pregunta': '¿qué es el ojo, cuál es su función y cómo funciona?',
 'relaciones': ['rel

Almacenamos en un archivo JSON el grafo de conocimiento obtenido y lo visalizamos en un archivo HTML 😀. Para ver el grafo descarguen el archivo `mapas.html` y abranlo en una nueva pestaña de su navegador:

In [47]:
guardar_mapas_conceptuales('./mapas.json', [mapa_conceptual])
dibujar_mapas_conceptuales('./mapas.html', [mapa_conceptual])

# Tarea

La tarea consiste en diseñar e implementar una estrategia propia para la creación de mapas conceptuales utilizando el modelo de lenguaje Gemini. Todo el proceso debe integrarse en una función llamada crear_mapa_conceptual, la cual recibe como entrada un texto y devuelve como salida un diccionario de Python que codifica el mapa conceptual correspondiente.

A partir del archivo `documentos.json`, deberán generar un único mapa conceptual por cada texto, y almacenar todos ellos en un único archivo JSON `mapas.json`, siguiendo el formato ilustrado en el ejemplo anterior.

Se recomienda experimentar con distintas estrategias de prompting y diferentes algoritmos. Pueden usar como referencia los prompts y piezas de código presentados en la sección previa. Además, el siguiente recurso puede serles útil como guía para diseñar prompts:
https://www.promptingguide.ai/

**Consejo**:
* Incorporen un paso adicional en la estrategia anterior que le pida al LLM mejorar un mapa conceptual previamente generado.

---

### Entrega

Cuando finalicen, suban su notebook, los prompts utilizados y todos los archivos con los mapas conceptuales generados a un repositorio público de GitHub. Envíen el enlace del repositorio al correo mat.torreta@gmail.com con el asunto:

* *InforSanLuis25-LLMs Entrega*

En el cuerpo del correo deben incluir:

* Nombre completo

* DNI

La entrega deberá realizarse antes del viernes 05 de diciembre a las 23:59 para aprobar el curso.

---

### Método a implementar

Implementen la mayor parte de su estrategia de creación de mapas conceptuales en el cuerpo de esta función. Pueden crear sus propias funciones auxiliares e invocarlas desde aquí si lo necesitan.

In [30]:
# Prompt de mejora
mejorar_mapa_sistema = f"""Eres un asistente experto en construcción y mejora de mapas conceptuales.
Recibes un mapa conceptual preliminar generado por otro modelo y debes
corregirlo y mejorarlo sin agregar información nueva que no esté en el texto
original utilizado para generarlo.
Tu trabajo consiste en limpiar, reorganizar y perfeccionar el mapa conceptual,
respetando estrictamente el contenido del texto fuente. No puedes inventar
hechos, relaciones ni conceptos.

TAREAS A REALIZAR:
1. Eliminar conceptos que no aparezcan en ninguna terna.
2. Eliminar relaciones que no aparezcan en ninguna terna.
3. Identificar nodos aislados (conceptos sin conexiones) y conectarlos
   agregando exactamente UNA terna adicional, basada únicamente en el texto
   original. Nunca inventar nueva información.
4. Corregir relaciones mal formadas, ambiguas, repetidas o demasiado
   genéricas. Si dos relaciones son duplicadas (texto exactamente igual),
   mantener solo una.
5. Corregir conceptos o relaciones que estén mal escritos, inconsistentes
   o que rompan la estructura semántica, siempre sin inventar hechos nuevos.
6. Eliminar ternas duplicadas. Si dos ternas son idénticas, conservar solo una.
7. Mantener la estructura estricta de cada terna:
   <concepto> @ <relación> @ <concepto>
8. No agregar nuevos conceptos ni nuevas relaciones que no estén implícitamente
   presentes en el mapa original o en el texto fuente.
9. Entregar el resultado FINAL únicamente en el siguiente formato:

CONCEPTOS:
- <concepto>

RELACIONES:
- <relación>

TERNAS:
@! <concepto> @ <relación> @ <concepto> !@

EJEMPLOS DE FORMATO (NO AGREGAR AL RESULTADO FINAL):

@
CONCEPTOS:
- Sistema Operativo
- Procesos
- Memoria
- Scheduler
@
RELACIONES:
- maneja
- controla
@
TERNAS:
@! Sistema Operativo @ maneja @ Procesos !@
@! Sistema Operativo @ maneja @ Memoria !@

@
CONCEPTOS:
- Aprendizaje
- Estudiante
- Conocimiento
@
RELACIONES:
- produce
- genera
@
TERNAS:
@! Aprendizaje @ produce @ Conocimiento !@
@! Estudiante @ genera @ Aprendizaje !@

Entrega únicamente el mapa conceptual corregido en el formato indicado.
"""

In [31]:
# Recibe el titulo y el texto
def crear_mapa_conceptual(texto: str) -> dict:
    """
    IMPORTANTE!

    Aquí deben implementar la estrategia para crear mapas conceptuales
    a partir de texto. Deberán coordinar las diferentes invocaciones
    al modelo de lenguaje y procesar correctamente las respuestas.
    """
    # print(texto)

    # Paso 1 : Genera la pregunta focal
    instruccion_crear_pregunta_formateada = crear_pregunta_usuario.format(texto = texto)
    preguntaFocalTexto = invocar_llm( usuario = instruccion_crear_pregunta_formateada, sistema = crear_pregunta_sistema)
    preguntaFocalTexto = normalizar_etiqueta(preguntaFocalTexto)

    # Paso 2 : Genera la lista de conceptos
    instruccionCrearConceptosFormateada = crear_conceptos_usuario.format(pregunta = preguntaFocalTexto, texto = texto)
    respuestaLLM = invocar_llm( usuario = instruccionCrearConceptosFormateada, sistema = crear_conceptos_sistema)
    conceptos_exp_reg = r'[\w\d].*?\n|[\w\d].*$'
    lista_conceptos = re.findall(conceptos_exp_reg, respuestaLLM)
    lista_conceptos = [normalizar_etiqueta(concepto) for concepto in lista_conceptos]
    # pprint.pprint(lista_conceptos)

    # Paso 3 : Creacion de la lista de relaciones sematicas
    instruccion_crear_relaciones_formateada = crear_relaciones_usuario.format( pregunta = preguntaFocalTexto, conceptos = "\n".join(lista_conceptos), texto = texto)
    respuesta_llm = invocar_llm( sistema = crear_relaciones_sistema, usuario = instruccion_crear_relaciones_formateada)
    relaciones_exp_reg = r'[\w\d].*?\n|[\w\d].*$'
    lista_relaciones = re.findall(relaciones_exp_reg, respuesta_llm)
    lista_relaciones = [normalizar_etiqueta(relacion) for relacion in lista_relaciones]
    # pprint.pprint(lista_relaciones)

    # Paso 4 : Lista de ternas de conocimiento
    instruccion_crear_ternas_formateada = crear_ternas_usuario.format( pregunta = preguntaFocalTexto + "\n", conceptos = "\n".join(lista_conceptos) + "\n",
                                                                       relaciones = "\n".join(lista_relaciones), texto = texto)
    respuesta_llm = invocar_llm( sistema = crear_ternas_sistema, usuario = instruccion_crear_ternas_formateada)
    ternas_exp_reg = r'@! (.+?) @ (.+?) @ (.+?) !@'
    L = re.findall(ternas_exp_reg, respuesta_llm)
    listaTernas = []
    for f, r, o in L:
        listaTernas.append({ 'f': normalizar_etiqueta(f), 'r': normalizar_etiqueta(r), 'o': normalizar_etiqueta(o) })

    # Paso 5: Mejora del mapa
    # Base para el mapa mejorado
    mapa_base = {
        'pregunta': preguntaFocalTexto,
        'conceptos': lista_conceptos,
        'relaciones': lista_relaciones,
        'ternas': listaTernas
    }

    # Formatear la entrada para el LLM de mejora
    entrada_mejora = f"""Mapa preliminar:
                    CONCEPTOS:
                    {chr(10).join(['- ' + c for c in lista_conceptos])}

                    RELACIONES:
                    {chr(10).join(['- ' + r for r in lista_relaciones])}

                    TERNAS:
                    {chr(10).join([f"@! {t['f']} @ {t['r']} @ {t['o']} !@" for t in listaTernas])}

                    Texto fuente:
                    {texto}
                    """

    # Invocar LLM con prompt de mejora
    respuesta_mejorada = invocar_llm(
        usuario=entrada_mejora,
        sistema=mejorar_mapa_sistema
    )

    # Dividir el texto usando los encabezados
    partes = respuesta_mejorada.split("RELACIONES:")
    bloque_conceptos = partes[0]
    resto = partes[1] if len(partes) > 1 else ""

    partes_ternas = resto.split("TERNAS:")
    bloque_relaciones = partes_ternas[0]
    bloque_ternas = partes_ternas[1] if len(partes_ternas) > 1 else ""

    # Ahora aplicamos regex solo en su bloque correspondiente
    conceptos_mejorada = re.findall(r'-\s+(.+)', bloque_conceptos)
    conceptos_mejorada = [normalizar_etiqueta(c.strip()) for c in conceptos_mejorada]
    relaciones_mejorada = re.findall(r'-\s+(.+)', bloque_relaciones)
    relaciones_mejorada = [normalizar_etiqueta(r.strip()) for r in relaciones_mejorada]
    ternas_mejorada_raw = re.findall(r'@! (.+?) @ (.+?) @ (.+?) !@', bloque_ternas)
    listaTernas_mejorada = []

    for f, r, o in ternas_mejorada_raw:
        listaTernas_mejorada.append({
            'f': normalizar_etiqueta(f),
            'r': normalizar_etiqueta(r),
            'o': normalizar_etiqueta(o)
        })

    # Retornar mapa mejorado
    return {
        "pregunta": preguntaFocalTexto,
        "conceptos": conceptos_mejorada,
        "relaciones": relaciones_mejorada,
        "ternas": listaTernas_mejorada
    }

    # raise RuntimeError("Función no implementada.")

In [32]:
mapas = []

for i in range(len(documentos)):
    texto = documentos[i]['texto']
    mapa = crear_mapa_conceptual(texto)

    mapaConceptual = {
        'titulo': documentos[i]['titulo'],
        **mapa
    }

    mapas.append(mapaConceptual)

guardar_mapas_conceptuales("mapas2.json", mapas)
dibujar_mapas_conceptuales("mapas2.html", mapas)

# Bibliografía

* [Unifying Large Language Models and Knowledge Graphs: A Roadmap](https://arxiv.org/abs/2306.08302) de Shirui Pan, et al.
* [The Theory Underlying Concept Maps and How
to Construct and Use Them](https://cmap.ihmc.us/publications/researchpapers/theoryunderlyingconceptmaps.pdf) de Joseph D. Novak y Alberto J. Cañas.